# ExTetris Variant Designer Studio

Design your own ExTetris game **without programming**:

1. **Run each cell from top to bottom** (Shift+Enter, or the play button).
2. Change only the values in the **form fields**: lines ending in `# @param`. In Google Colab they appear as menus and sliders; in Jupyter, just edit the value after the `=`.
3. At the end, the notebook checks your game and saves it as a `.toml` file you can play with
   `python v2_platform_release/cli.py play variants/<your_file>.toml`.

Nothing you type here is run as game code: every setting is checked by the same validator as the command-line tool.

In [ ]:
# Setup: finds the ExTetris folders. Run this cell first; nothing to edit.
import sys
from pathlib import Path
here = Path.cwd()
root = next(p for p in [here, *here.parents] if (p / "v2_platform_release" / "spec.py").exists())
sys.path.insert(0, str(root / "v2_platform_release"))
from spec import SpecValidator, VariantSpec, dump_toml          # also makes v1_core_system importable
from family import PieceFamily
from simulator import simulate
try:
    import matplotlib.pyplot as plt
    HAVE_PLOT = True
except ImportError:
    HAVE_PLOT = False
print("Ready. Project folder:", root, "| pictures:", "matplotlib" if HAVE_PLOT else "text only")

## Step 1 - Choose your pieces
`Classic` = the 7 original tetrominoes. `Extended-Easy` = 15 pieces that can land leaving at most one hole. `Full` = all 34 pieces made of 4 squares touching by edges or corners.

In [ ]:
preset = "Extended-Easy"  # @param ["Classic", "Extended-Easy", "Full"]
remove_pieces = ""  # @param {type:"string"}
add_pieces = ""  # @param {type:"string"}

def _names(text):
    return [t.strip() for t in text.replace(",", " ").split() if t.strip()]

pieces_section = {"preset": preset}
if _names(remove_pieces):
    pieces_section["exclude"] = _names(remove_pieces)
if _names(add_pieces):
    pieces_section["include"] = _names(add_pieces)
problems = SpecValidator({"pieces": pieces_section})
if problems.validate():
    family = VariantSpec.from_dict({"pieces": pieces_section}).family()
    print(f"{len(family)} pieces selected:", ", ".join(p.name for p in family))
else:
    print("Please fix:", *problems.errors, sep="\n  - ")

## Step 2 - Look at your pieces

In [ ]:
def show_pieces(fam):
    pieces = list(fam)
    if not HAVE_PLOT:
        for p in pieces:
            print(p.name, "\n" + str(p), "\n")
        return
    cols = 8
    rows = (len(pieces) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.4, rows * 1.6))
    for ax in getattr(axes, "flat", [axes]):
        ax.axis("off")
    for ax, p in zip(getattr(axes, "flat", [axes]), pieces):
        ax.imshow(~p.to_matrix(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(p.name, fontsize=7)
    plt.tight_layout()
    plt.show()

show_pieces(family)

## Step 3 - How often each piece appears
- `uniform`: all equal
- `bag`: every piece once per round, in shuffled order
- `by_property`: favour pieces by a rule
- `adaptive`: pieces you haven't seen for a while become more likely

In [ ]:
kind = "adaptive"  # @param ["uniform", "bag", "by_property", "adaptive"]
rule = "favor_few_holes"  # @param ["uniform", "favor_few_holes", "favor_classic", "favor_small_bbox", "favor_symmetric"]
bag_copies = 1  # @param {type:"slider", min:1, max:4, step:1}
drought_boost = 2.0  # @param {type:"slider", min:0, max:5, step:0.5}

distribution_section = {"kind": kind}
if kind == "by_property":
    distribution_section["rule"] = rule
elif kind == "bag":
    distribution_section["copies"] = int(bag_copies)
elif kind == "adaptive":
    distribution_section.update({"base": "by_property", "rule": rule, "drought_boost": float(drought_boost)})

preview = VariantSpec.from_dict({"pieces": pieces_section, "distribution": distribution_section})
probs = preview.build_engine(seed=0).distribution.probabilities()
names = [p.name for p in preview.family()]
values = [100 * probs[p.code] for p in preview.family()]
if HAVE_PLOT:
    plt.figure(figsize=(max(6, len(names) * 0.35), 3))
    plt.bar(range(len(names)), values)
    plt.xticks(range(len(names)), names, rotation=90, fontsize=7)
    plt.ylabel("chance (%)")
    plt.title("Chance of each piece at the start of a game")
    plt.tight_layout()
    plt.show()
else:
    for n, v in zip(names, values):
        print(f"{n:14s} {'#' * int(v * 2)} {v:.1f}%")

## Step 4 - Board, speed and physics

In [ ]:
game_name = "My First Variant"  # @param {type:"string"}
author = "me"  # @param {type:"string"}
width = 10  # @param {type:"slider", min:6, max:20, step:1}
height = 20  # @param {type:"slider", min:12, max:30, step:1}
walls = "solid"  # @param ["solid", "wrap"]
gravity = "naive_drop"  # @param ["naive_drop", "cascade"]
start_speed_ms = 1000  # @param {type:"slider", min:200, max:2000, step:50}
speed_up_factor = 0.9  # @param {type:"slider", min:0.8, max:1.0, step:0.01}
lines_per_level = 10  # @param {type:"slider", min:2, max:30, step:1}
next_pieces_shown = 3  # @param {type:"slider", min:1, max:6, step:1}
theme = "dark"  # @param ["dark", "light", "retro"]

variant = {
    "variant": {"name": game_name, "version": "1.0", "authors": [author],
                "description": "Made with the Variant Designer Studio."},
    "board": {"width": int(width), "height": int(height), "walls": walls},
    "pieces": pieces_section,
    "distribution": distribution_section,
    "levels": {"base_ms": int(start_speed_ms), "factor": float(speed_up_factor),
               "lines_per_level": int(lines_per_level), "min_ms": 60},
    "physics": {"gravity": gravity, "preview": int(next_pieces_shown)},
    "theme": {"name": theme},
}
check = SpecValidator(variant)
if check.validate():
    print("Your variant is valid.", *check.warnings, sep="\n  note: ")
else:
    print("Please fix these settings:", *check.errors, sep="\n  - ")

## Step 5 - Balance check
A computer player plays some games with your settings. Very short games mean the variant is hard; games that always hit the piece limit mean it is easy.

In [ ]:
games = 10  # @param {type:"slider", min:5, max:100, step:5}
report = simulate(VariantSpec.from_dict(variant), games=int(games), max_pieces=200)
print(report.summary())
if HAVE_PLOT:
    plt.figure(figsize=(6, 2.5))
    plt.hist(report.pieces, bins=10)
    plt.xlabel("pieces played before game over")
    plt.ylabel("games")
    plt.tight_layout()
    plt.show()

## Step 6 - Save your game

In [ ]:
file_name = "my_first_variant"  # @param {type:"string"}
target = root / "variants" / (file_name.strip().replace(" ", "_") + ".toml")
target.parent.mkdir(exist_ok=True)
target.write_text(dump_toml(variant, "# Made with the ExTetris Variant Designer Studio\n"), encoding="utf-8")
VariantSpec.from_toml(target)       # read it back to be sure it is valid
print("Saved", target)
print("Play it:     python v2_platform_release/cli.py play", target.relative_to(root).as_posix())
print("Package it:  python v2_platform_release/cli.py package", target.relative_to(root).as_posix())